# PDF Extraction Test — FPT ESG Report 2023

Tests `extraction/pdf_parser.py` → `extract_and_save()` against the FPT ESG 2023 PDF.

Covers:
- Text block extraction
- Table extraction (with header, rows, raw)
- Image extraction (with content_type_hint)
- Output JSON structure validation

In [ ]:
import sys
import json
from pathlib import Path

# Add repo root to path so we can import extraction/
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from extraction.pdf_parser import PDFParser

PDF_PATH   = REPO_ROOT / "data/raw/crawled_annual_report/2023-fpt-esg-report.pdf"
OUTPUT_DIR = REPO_ROOT / "output/extracted"

print("PDF    :", PDF_PATH)
print("Exists :", PDF_PATH.exists())
print("Output :", OUTPUT_DIR)

## 1. Run Extraction

In [ ]:
parser = PDFParser(min_text_length=50, extract_tables=True)
result = parser.extract_and_save(PDF_PATH, OUTPUT_DIR)

doc  = result["document"]
blks = result["blocks"]

print(f"Document ID  : {doc['document_id']}")
print(f"Source file  : {doc['source_file']}")
print(f"Extracted at : {doc['extracted_at']}")
print(f"Total pages  : {doc['total_pages']}")
print(f"Total blocks : {len(blks)}")

## 2. Block Type Summary

In [ ]:
from collections import Counter

type_counts = Counter(b["type"] for b in blks)
numeric_counts = {
    t: sum(1 for b in blks if b["type"] == t and b["metadata"].get("has_numeric_data"))
    for t in type_counts
}

print(f"{'Type':<10} {'Count':>6}  {'Has numeric data':>18}")
print("-" * 38)
for t, cnt in sorted(type_counts.items()):
    print(f"{t:<10} {cnt:>6}  {numeric_counts.get(t, 0):>18}")

## 3. Inspect: Text Blocks

In [ ]:
text_blocks = [b for b in blks if b["type"] == "text"]

# Show first 3 text blocks
for blk in text_blocks[:3]:
    print(f"ID            : {blk['id']}")
    print(f"Page          : {blk['page']}")
    print(f"bbox          : {blk['layout']['bbox']}")
    print(f"reading_order : {blk['layout']['reading_order']}")
    print(f"has_numeric   : {blk['metadata']['has_numeric_data']}")
    print(f"Text preview  : {blk['content']['text'][:300]!r}")
    print(f"Context after : {blk['context']['after'][:100]!r}")
    print("-" * 70)

## 4. Inspect: Table Blocks

In [ ]:
table_blocks = [b for b in blks if b["type"] == "table"]

if not table_blocks:
    print("No tables extracted from this PDF.")
else:
    # Show first 3 tables
    for blk in table_blocks[:3]:
        print(f"ID            : {blk['id']}")
        print(f"Page          : {blk['page']}")
        print(f"bbox          : {blk['layout']['bbox']}")
        print(f"reading_order : {blk['layout']['reading_order']}")
        print(f"has_numeric   : {blk['metadata']['has_numeric_data']}")
        print(f"Header        : {blk['content']['header']}")
        print(f"Rows (first 3):")
        for row in blk['content']['rows'][:3]:
            print(f"  {row}")
        print(f"Raw (first 2 lines):")
        for line in blk['content']['raw'].split('\n')[:2]:
            print(f"  {line}")
        print(f"Context before: {blk['context']['before'][:100]!r}")
        print("-" * 70)

## 5. Inspect: Image Blocks

In [ ]:
from IPython.display import Image as IPImage, display

image_blocks = [b for b in blks if b["type"] == "image"]

if not image_blocks:
    print("No images extracted (PyMuPDF may not be installed).")
else:
    print(f"Total images: {len(image_blocks)}\n")

    # Count by content_type_hint
    hint_counts = Counter(b["metadata"].get("content_type_hint", "unknown") for b in image_blocks)
    print("Content type hints:", dict(hint_counts))
    print()

    # Show first 5 image blocks and render them
    for blk in image_blocks[:5]:
        print(f"ID            : {blk['id']}")
        print(f"Page          : {blk['page']}")
        print(f"bbox          : {blk['layout']['bbox']}")
        print(f"reading_order : {blk['layout']['reading_order']}")
        print(f"content_type  : {blk['metadata']['content_type_hint']}")
        print(f"format        : {blk['content']['format']}")
        img_path = OUTPUT_DIR / blk['content']['path']
        print(f"path          : {img_path}")
        if img_path.exists():
            display(IPImage(filename=str(img_path), width=400))
        else:
            print("  [image file not found on disk]")
        print("-" * 70)

## 6. Validate JSON Structure

Check that every block conforms to the expected schema before KG ingestion.

In [ ]:
REQUIRED_COMMON = {"id", "type", "page", "content", "layout", "context", "metadata"}
REQUIRED_LAYOUT = {"bbox", "reading_order"}
TYPE_CONTENT_KEYS = {
    "text":  {"text"},
    "table": {"rows", "header", "raw"},
    "image": {"path", "format"},
}
TYPE_META_KEYS = {
    "text":  {"has_numeric_data", "extraction_method"},
    "table": {"has_numeric_data", "extraction_method"},
    "image": {"content_type_hint", "extraction_method"},
}

errors = []
for blk in blks:
    bid = blk.get("id", "?")
    btype = blk.get("type", "?")

    # Common top-level keys
    missing = REQUIRED_COMMON - blk.keys()
    if missing:
        errors.append(f"{bid}: missing top-level keys {missing}")

    # Layout keys
    layout = blk.get("layout", {})
    missing = REQUIRED_LAYOUT - layout.keys()
    if missing:
        errors.append(f"{bid}: missing layout keys {missing}")

    # bbox must be 4 floats
    bbox = layout.get("bbox", [])
    if len(bbox) != 4 or not all(isinstance(v, (int, float)) for v in bbox):
        errors.append(f"{bid}: invalid bbox {bbox!r}")

    # Content keys by type
    expected_content = TYPE_CONTENT_KEYS.get(btype, set())
    content = blk.get("content", {})
    missing = expected_content - content.keys()
    if missing:
        errors.append(f"{bid}: missing content keys {missing}")

    # Metadata keys by type
    expected_meta = TYPE_META_KEYS.get(btype, set())
    meta = blk.get("metadata", {})
    missing = expected_meta - meta.keys()
    if missing:
        errors.append(f"{bid}: missing metadata keys {missing}")

    # Context must have before + after
    ctx = blk.get("context", {})
    if "before" not in ctx or "after" not in ctx:
        errors.append(f"{bid}: missing context before/after")

if errors:
    print(f"VALIDATION FAILED — {len(errors)} error(s):")
    for e in errors[:20]:
        print(f"  {e}")
else:
    print(f"All {len(blks)} blocks passed schema validation.")

## 7. Output JSON File

Verify the saved file and show its size and a sample block.

In [ ]:
json_path = OUTPUT_DIR / doc["document_id"] / "extraction_result.json"
size_kb = json_path.stat().st_size / 1024

print(f"JSON path : {json_path}")
print(f"File size : {size_kb:.1f} KB")
print()

# Reload from disk to confirm round-trip
with open(json_path, encoding="utf-8") as f:
    loaded = json.load(f)

assert loaded["document"]["document_id"] == doc["document_id"], "document_id mismatch after reload"
assert len(loaded["blocks"]) == len(blks), "block count mismatch after reload"
print("Round-trip reload: OK")
print()

# Print a sample table block (if any) from the loaded JSON
sample = next((b for b in loaded["blocks"] if b["type"] == "table"), None)
if sample:
    print("Sample table block (from disk):")
    print(json.dumps(sample, ensure_ascii=False, indent=2)[:800])
else:
    sample = next((b for b in loaded["blocks"] if b["type"] == "text"), None)
    print("Sample text block (from disk):")
    preview = dict(sample)
    preview["content"] = {"text": sample["content"]["text"][:200] + "..."}
    print(json.dumps(preview, ensure_ascii=False, indent=2))